In [ ]:
import numpy as np
import tempfile
import os
import matplotlib.pyplot as plt
import scipy.io as sci
from munkres import Munkres
from scipy.io import loadmat

Ce TP traite de la partie du cours sur l'imagerie hyperspectrale. Vous devez rendre une version complétée du notebook d'ici une semaine.

# I - Hyperspectral unmixing

# I - 0 - Introduction: original motivation for the MU algorithm: dimensionality reduction

As an introduction, we will first use NMF as a dimensionality reduction tool, which corresponds to the orginal motivation of the Multiplicative Update (MU) algorithm. Specifically, in this whole part, we will try to extract features from a data set composed of pictures. In contrast to hyperspectral unmixing, we will not look for "true" features at the origin of the data set, but we will merely perform dimensionality reduction.

## I - 0 - 1) Dataset

Load the original images present in the files *'YaleB\_32x32.mat'*. This is a small part of the freely available Extended Yale Face Database B downloaded from http://www.cad.zju.edu.cn/home/dengcai/Data/FaceData.html. It contains 2414 cropped images resized to 32x32 pixels. Every image is represented as a vector 1x1024 and all images are stacked in a matrix called data. There are 38 subjects with around 64 near frontal images per individual under different illumination conditions. Once loaded and normalised the data, such that the pixels are between 0 and 1, you can plot images.

**Goal**

The goal of this part is to evaluate qualitatively the relevance of the features found by NMF. To do so, we above load the dataset and implement a function to plot the faces.

In [ ]:
# Please modify working_dir if required
Working_directory = "./"


x = loadmat(Working_directory + "./YaleB_32x32.mat")
data = x["fea"]
d = data.shape[1]  # number of pixels of the images
subjectIndex = x["gnd"]  # we have one index per subject
maxValue = np.max(np.max(data))  # max intensity value
data = data / maxValue  # Scale pixels to [0,1]

Ns = len(np.unique(subjectIndex))  # Number subjects
Is = round(
    len(subjectIndex) / Ns
)  # Number images per subject (on average, not the same number for every subject)
r = int(np.sqrt(d))  # number rows of each image
c = r  # number columns of each image, equal to row since images are square

print("There are", data.shape[0], "facial images and each image has", d, "pixels")
print(
    "There are", Ns, "different subjects and each subject has on average", Is, "images"
)

X = data


def plotFaces(data, r, c, ncol=2, N=0, indeces=None, title=None):
    # data: each face is a row in data
    # r,c = number of rows and columns of each image
    # n_col = number of columns for subplots
    # N = random images to plot (used only if indeces is empty)
    # indeces = indeces of images to plot
    # title = title of the plot

    if indeces is None:
        if N == 0:
            raise NameError("You should define either N or indeces")
        else:
            print("Use N random subjects")
            indeces = np.random.randint(0, data.shape[0], (N, 1))

    nrow = int(np.ceil(len(indeces) / ncol))

    fig = plt.figure(figsize=(17, 6))
    plt.suptitle(title, size=16)
    for i, index in enumerate(indeces):
        fig.add_subplot(nrow, ncol, i + 1)
        plt.imshow(np.resize(data[index, :], (r, c)).T, origin="upper", cmap="gray")
        plt.xticks(())
        plt.yticks(())

## I - 0 - 2) MU algorithm

Here you will implement the MU algorithm and launch it to reduce the dimensionality of the dataset. In this context, the column of $A$ will correspond to basis images and the coefficients of $S$ will correspond to some decomposition coefficients in the $A$ basis.

**Question**

1) Complete below the codes where XXXXXXX are written.
2) Plot the basis images found by NMF/ What can you say ?

__answer__: 
The matrix 
𝐴
represents the fundamental components that best approximate the data matrix 
𝑋. Given that the dataset consists mainly of facial parts, these components correspond to recurring features across the dataset. Since such components are consistently present, the algorithm appears to work effectively by extracting these common patterns.

In [ ]:
def MULecture(
    X, n=None, N_Iter=1000, tolerance=1e-3, plot_evolution=1, A=-1, S=-1, frozenA=False
):
    """
    Inputs:
    %           X: is a [mxt] matrix to unmix
    %
    %           n: number of columns of A (= number of rows of S)
    %
    %           (Optional) N_Iter: maximum number of iterations
    %
    %           (Optional) tolerance: convergence criteria threshold
    %
    %           (Optional) plot_evolution: plot evolution convergence criteria
    %
    %           (Optional) A,S = initialization of A and S
    %
    %           (Optional) frozenA (default value = False): set True to update only S (A is kept as specified in its initialization)
    %
    % Outputs:
    %           A: is a [m x n] matrix
    %
    %           S: is a [n x t] matrix
    %
    """
    if n is None:
        n = X.shape[0]

    if frozenA:
        n = A.shape[1]

    # Test for positive values
    if np.min(X) < 0:
        raise NameError("Input matrix X has negative values !")

    # Size
    d, N = X.shape

    # Initialization
    if np.any(A < 0) and (not frozenA):
        # Chose a random positive matrix
        A = np.random.rand(d, n)
    if np.any(S < 0):
        S = np.random.rand(n, N)

    # parameters for convergence
    k = 0
    delta = np.inf
    eps = np.finfo(float).eps
    evolutionDelta = []

    while delta > tolerance and k < N_Iter:
        # Multiplicative method
        S = S * (A.T @ X) / (A.T @ A @ S + eps)

        if not frozenA:
            Xs = np.dot(X, S.T)
            SS = np.dot(S, S.T)

        # Update A (if not frozen)
        if not frozenA:
            A = A * (X @ S.T) / (A @ (S @ S.T) + eps)

            # Add a +eps in the denominator to avoid division by 0

        # Add a +eps in the denominator to avoid division by 0

        # Convergence indices
        k = k + 1
        diff = X - np.dot(A, S)

        delta = np.linalg.norm(diff, "fro") / np.linalg.norm(X, "fro")
        evolutionDelta.append(delta)

        if k == 1 or k % 100 == 0:
            print(
                "Iteration NNMF number ",
                k,
                " out of ",
                N_Iter,
                ", delta = ",
                delta,
                ", error (norm delta): ",
                np.linalg.norm(diff),
            )

    if k == N_Iter:
        print("Maximum number of iterations reached ! delta = ", delta)
    else:
        print("Convergence achieved ( delta = ", delta, ") in ", k, " iterations")

    if plot_evolution == 1:
        plt.figure(figsize=(6, 6))
        plt.plot(range(k), evolutionDelta, "bx--", linewidth=4, markersize=12)
        plt.title("Evolution of error - NNMF")
        plt.show()

    return A, S

In [ ]:
# Launch the MU algorithm
Ncomponents = 100
A, S = MULecture(X.T, n=Ncomponents, N_Iter=300, tolerance=1e-3, plot_evolution=1)
print(c)
plotFaces(A.T, r, c, ncol=2, indeces=np.arange(0, 10, 1), title="NNMF-faces")

# I - 1) Pure pixel NMF for hyperspectral unmixing
We now come back to hyperspectral images. This part of the practical work aims at performing hyperspectral unmixing. We will implement the VCA algorithm and the MU.

## I - 1 - 1) Data generation and manipulation

In the remaining of this part about unmixing, we will use two datasets :
- a first simple one, to understand how the VCA iterations work ;
- a real hyperspectral one, called urban.
Urban is one of the most widely used hyperspectral data used in the hyperspectral unmixing study. There are 307 x 307 pixels, each of which corresponds to a 2 x 2 m2 area. There are 210 wavelengths ranging from 400 nm  to 2500 nm, resulting in a spectral resolution of 10 nm. After the channels 1--4, 76, 87, 101--111, 136--153 and 198--210 are removed (due to dense water vapor and atmospheric effects), we obtain 162  channels.
Interestingly enough, a ground truth have been established. The one we will use contains 6 sources.

**Simple dataset**

We propose to generate a very simple dataset, for visual purposes, as follows:
- the mixing matrix **A** will be a m x n matrix with its coefficients generated randomly in [0,10]. When you obtain such a realisation, verify that **A** is not too badly conditioned.
- the source matrix **S** will be a n x t matrix with random coefficients in [0,1] but will be scalled so that the l1-norm of each of its columns sum to 1.
- There will be no noise.

In [ ]:
# TO DO : compute the condition number of A. What would be the issue with an ill-conditioned mixing matrix?

n = 3
m = 3
t = 500

A_toy = np.random.rand(m, n) * 10

S_toy = np.random.rand(n, t)
S_toy = S_toy / np.sum(S_toy, axis=0)

X_toy = np.dot(A_toy, S_toy)

# assert A is not ill conditioned
A_toy_inv = np.linalg.pinv(A_toy)


fig = plt.figure()
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_toy[0, :], X_toy[1, :], X_toy[2, :])

**Real hyperspectral data set** 

Here, to matrices can be found :
- The data matrix **X = AS + N**, which has been acquired by a true sensor ;
- The groundtruth **S_gt**, which corresponds to the "true" abundances (aka concentration of each material). Note that having access to **S_gt** is exceptional: for most other datasets, it would be unknown.

In [ ]:
data = sci.loadmat("Urban.mat")
X = data["X"]
X = X.astype(
    float
)  # Please take care that the data matrix must be cast to float in Python, for subsequential operations

gt = sci.loadmat("end6_groundTruth.mat")
abundances = gt["A"]

nCol = 307
nRow = 307

plt.figure()
names = ["Road", "Grass", "Trees", "Rooftop (2)", "Rooftop (1)", "Dirt"]
for ii in range(6):
    ax = plt.subplot(2, 3, ii + 1)
    ax.imshow(abundances[ii, :].reshape(nCol, nRow))

    plt.title(names[ii])

__answer__:
- TO DO : is the near separable (aka pure pixel) assumption fulfilled on this dataset? Explain why.


The dataset consists of distinct land cover classes such as roads, grass, trees, rooftops, and dirt. Each pixel is predominantly composed of one land cover type (e.g., a road, grass, or rooftop). These classes are spatially distinct, making the near-separable (pure pixel) assumption valid for this dataset.
- TO DO : we have access to the abundance **S_gt** groundtruth but not to the endmembers **A_gt**. Use your answer to the above question to explain how the following code finds the columns of the mixing matrix A. 


Since each pixel predominantly belongs to one class, the code identifies the pixels with the highest abundances for each class. It then averages the spectral signatures of these pure pixels to estimate the endmembers for the corresponding classes, which represent the columns of the mixing matrix A.








In [ ]:
endmembers = np.zeros((162, 6))

for ii in range(6):
    ind = np.where(abundances[ii] > 0.999)[0]
    allPP = X[:, ind]
    endmembers[:, ii] = np.mean(allPP, axis=1)

ind = np.where(abundances[0] > 0.999)[0]
allPP = X[:, ind]
plt.figure(), plt.plot(allPP)

In [ ]:
plt.figure(), plt.plot(endmembers)

# I - 1 - 2) VCA

We will here implement the VCA pure-pixel NMF algorithm and look at its practical efficiency.

1) Recall what pure-pixel NMF is.
__answer__:
In the context of hyperspectral imaging or similar applications, pure-pixel NMF assumes that each pixel's spectral signature comes from one pure material or class. The algorithm decomposes the data matrix \( V \) into two non-negative matrices: one matrix \( A \) that contains the endmembers (pure components) and one matrix \( S \) that contains the corresponding abundance coefficients, which represent how much each endmember contributes to each pixel.


2) Implement the VCA algorithm

In [ ]:
def simpleVCA(
    X, r, optDisp=False
):  # TO DO. NB : simple parce qu'on gère mal les cas d'égalité (on peut gagner environ 0.06 sur l'angle)
    R = X
    m = X.shape[0]

    K = np.zeros(r)

    for ii in range(r):
        c = np.random.randn(m)
        ctX = c.T @ R

        p = np.argmax(ctX)
        K[ii] = p
        Rp = np.expand_dims(R[:, p], axis=1)

        if optDisp == True:
            fig = plt.figure()
            ax = fig.add_subplot(111, projection="3d")
            ax.scatter(R[0, :], R[1, :], R[2, :])

        R = (np.eye(m) - np.dot(Rp, Rp.T) / np.linalg.norm(Rp) ** 2) @ R
        print("the residual norm is %s" % np.linalg.norm(R))
    print("Max residual %s" % np.max(R))

    return K.astype(int)

3) Launch the VCA algorithm on the toy example.

Plot the residual at each iteration of VCA. 

How many sources can you at most extract ? Why ?


__answer__: We can extract at most tje rank of the dataset. As we extract one random vector from the dataset that is linearly independent from the previous vectors. We can therefore extract at most the rank of the data sources  

In [ ]:
K_VCA = simpleVCA(X_toy, 3, optDisp=True)

**4)** The code below enables to compute a separation metric to measure the separation accuracy of the unmixing algorithms. 

Use it to assess the quality of VCA on the real dataset.
__answer__: Based on the plot, each class is correctly identified by VCA. The crops are estimated as "Trees," and the dirt is accurately represented as the "Dirt" class.


Plot the endmembers found by your algorithm. How good are they?

__answer__:
The endmembers found by the algorithm represent the spectral signatures of the pure components (such as trees, crops, dirt, etc.) in the dataset. Upon visualization, these endmembers should ideally resemble the unique spectral features corresponding to each material class. However, the spectral features are not clearly distinct, indicating that the algorithm's performance is average.


Try several run of VCA and comment the result stability.

__answer__: 

The problem is non-convex, which means that the optimization process can converge to different local minima depending on the initialization. As a result, running the VCA algorithm multiple times  produce different results, leading to different endmembers and abundance maps. The stability of the results will depend on the dataset's complexity and the VCA algorithm's initialization. This is why multiple runs of VCA  give varying plots, as shown below.


Do you have directly access to the abundance maps? How to obtain them?


__answer__:
We do not have direct access to the abundance maps. However, we can obtain them by applying the MU (Multiplicative Update) algorithm with the previously initialized endmembers. This will allow us to compute the abundance maps by fitting the data to the endmember matrix and estimating the abundance coefficients for each pixel.







In [ ]:
def norm_col(A):
    An = A.copy()
    type(An)
    for ii in range(np.shape(An)[1]):
        An[:, ii] = An[:, ii] / np.sqrt(np.sum(An[:, ii] ** 2))

    return An


def correctPerm(W0_en, W_en):
    # [WPerm,Jperm,err] = correctPerm(W0,W)
    # Correct the permutation so that W becomes the closest to W0.

    W0 = W0_en.copy()
    W = W_en.copy()

    W0 = norm_col(W0)
    W = norm_col(W)

    costmat = -W0.T @ W  # Avec Munkres, il faut bien un -

    m = Munkres()
    Jperm = m.compute(costmat.tolist())
    # print(Jperm)

    WPerm = np.zeros(np.shape(W0))
    indPerm = np.zeros(np.shape(W0_en)[1])

    for ii in range(W0_en.shape[1]):
        WPerm[:, ii] = W_en[:, Jperm[ii][1]]
        indPerm[ii] = Jperm[ii][1]

    return WPerm, indPerm.astype(int)


def evalCriterion(W0_en, W_en):
    # W0 : true mixing matrix
    # W : estimated mixing matrix
    #
    # maxAngle : cosine of the maximum angle between the columns of W0 and W

    W0 = W0_en.copy()
    W = W_en.copy()

    W, indPerm = correctPerm(W0, W)
    W0 = norm_col(W0_en)
    W = norm_col(W)

    diff = W0.T @ W
    return np.mean(np.diag(diff))

In [ ]:
K_VCA = simpleVCA(X, 6)

A_VCA = X[:, K_VCA]

Ac_VCA, indPerm = correctPerm(endmembers, A_VCA)

plt.figure(), plt.plot(Ac_VCA)
plt.title("Estimation by VCA")

print("Critere VCA : %s" % evalCriterion(endmembers, A_VCA))

In [ ]:
K_VCA = simpleVCA(X, 6)

A_VCA = X[:, K_VCA]

Ac_VCA, indPerm = correctPerm(endmembers, A_VCA)

plt.figure(), plt.plot(Ac_VCA)
plt.title("Estimation by VCA")

print("Critere VCA : %s" % evalCriterion(endmembers, A_VCA))

In [ ]:
K_VCA = simpleVCA(X, 6)

A_VCA = X[:, K_VCA]

Ac_VCA, indPerm = correctPerm(endmembers, A_VCA)

plt.figure(), plt.plot(Ac_VCA)
plt.title("Estimation by VCA")

print("Critere VCA : %s" % evalCriterion(endmembers, A_VCA))

# I - 2 - NMF THROUGH MU

In this part, we will perform NMF. To do that, you will re-use the MU algorithm that you implemented above.

    1) Launch the MU algorithm on the real dataset
    2) Can you find a better initialization than the random one ? Try it !
__answer__: On réutilise l'initialisation des algorithmes précédents!  avec l'initilisation trouvé à la partie précédente

In [ ]:
A_MU, S_MU = MULecture(X, n=6, N_Iter=300, tolerance=1e-3, plot_evolution=1)

In [ ]:
A_MU, S_MU = MULecture(X, n=6, N_Iter=300, tolerance=1e-3, plot_evolution=1, A=A_VCA)

In [ ]:
Ac, indPerm = correctPerm(endmembers, A_MU)

plt.figure(), plt.plot(Ac)

Sc = S_MU[indPerm, :]

print("Critere MU : %s" % evalCriterion(endmembers, A_MU))

# %%
plt.figure()
names = [
    "Estimated Road",
    "Estimated Grass",
    "Estimated Trees",
    "Estimated Rooftop (2)",
    "Estimated Rooftop (1)",
    "Estimated Dirt",
]
for ii in range(6):
    ax = plt.subplot(2, 3, ii + 1)
    ax.imshow(Sc[ii, :].reshape(nCol, nRow))

    plt.title(names[ii])

# II - SUPER-RESOLUTION

We now change the way we deal with the low spatial resolution of HSIs, since our objective is now to perform super-resolution. To do that, we will see two simple single image super-resolution algorithms, and two multi-images algorithms.

## 1) Data

We here consider the "chikusei" dataset, which contains an hyperspectral image. To be able to assess the quality of the algorithms we will developp, we will work on two synthetic images generated from the chikusei dataset. Precisely, the code below enables to simulate a HSI image by deteriorating by a factor 6 the number of pixels in the chikusei HSI, and to simulate a MSI by deteriorating the spectral content.

In [ ]:
from CNMF import gaussian_down_sample, zoom_bi, zoom_nn

In [ ]:
# Read image
filename = "chikusei.bsq"
cols1 = 240
rows1 = 240
bands1 = 128
data = np.transpose(
    np.double(
        np.fromfile(filename, dtype=np.uint16, count=cols1 * bands1 * rows1).reshape(
            bands1, rows1, cols1
        )
    ),
    (1, 2, 0),
)
hs_bands = np.r_[4:114]
data = data[:, :, hs_bands]  # Exclude noisy bands
bands1 = data.shape[2]

# Synthesize MSI
print("Synthesize MSI ...")
with open("chikusei_spec.txt", "r") as f:
    spec = map(lambda x: x.split(), f.read().strip().split("\n"))
spec = np.array([[float(elm) for elm in v] for v in spec])
wavelength = spec[hs_bands, 0]  # Center wavelength of HSI
RapidEye_wavelength = np.array(
    [[0.440, 0.510], [0.520, 0.590], [0.630, 0.690], [0.690, 0.730], [0.760, 0.880]]
)  # Spectral range of RapidEye
bands2 = RapidEye_wavelength.shape[0]  # Number of MSI bands
srf = np.zeros((bands2, bands1))
for b in range(bands2):
    b_i = np.nonzero(wavelength > RapidEye_wavelength[b, 0])[0][0]
    b_e = np.nonzero(wavelength < RapidEye_wavelength[b, 1])[0][-1]
    srf[b, b_i : b_e + 1] = 1.0 / (b_e + 1 - b_i)  # Rectangle SRF
MSI = np.dot(data.reshape(cols1 * rows1, bands1), srf.transpose()).reshape(
    rows1, cols1, bands2
)  # Spectral simulation

# Synthesize HSI
print("Synthesize HSI ...")
w = 6  # GSD difference
HSI = gaussian_down_sample(data, w)  # Spatial simulation via Gaussian filtering

data_res = data
HSI_res = HSI
MSI_res = MSI
data = data.transpose((2, 0, 1))
MSI = MSI.transpose((2, 0, 1))
HSI = HSI.transpose((2, 0, 1))

**Question:** how many spectral bands are there in the HSI? In the MSI? Same question for the spectral resolution.

__answer__: There is 110 Spectral band in the HSI and 5 in the MSI

In [ ]:
plt.figure(), plt.imshow(data[0, :, :]), plt.title("Ground-truth")
plt.figure(), plt.imshow(MSI[0, :, :]), plt.title("MSI")
plt.figure(), plt.imshow(HSI[0, :, :]), plt.title("HSI")

## 2) Simple bicubic interpolation (single image SR)

Now, let us first test a basic method: we will spatialy interpolate each spectral band of the HSI by a factor $w$ to increase the spatial resolution of the HSI.

Fill the gaps in the code below. You can use the function zoom_bi(im,w) for the interpolation, where im is a 2D image and w the interpolation factor.

In [ ]:
HSI_bilin = np.zeros((HSI.shape[0], HSI.shape[1] * w, HSI.shape[2] * w))

for ii in range(HSI.shape[0]):
    HSI_bilin[ii, :, :] = zoom_bi(HSI[ii, :, :], w)

Quantitative evaluation 1: in the cell below, we plot all the spectra associated to a line of the HR-HSI. Compare with the ground-truth in data. Do you see any difference?

__answer__: The estimation with the bicubic interpolation is smoother than the ground truth. This is an artifcat of the interpolation.


In [ ]:
plt.figure(), plt.plot(data[:, 120, :]), plt.title("Ground-truth")
plt.figure(), plt.plot(HSI_bilin[:, 120, :]), plt.title("Estimation")

Quantitative evaluation 2: now look at the spatial content of the interpolated HSI image. To do that, you can plot a few slices of HSI_bilin and compare with the ground-truth. What do you observe?

__answer__: The output is more blurry than the ground truth. We can observe that the output of the model is more continuous than the ground truth, which suggests that bicubic interpolation smooths out the high-frequency details present in the original data, resulting in a less sharp image.

In [ ]:
plt.figure(), plt.imshow(data[20, :, :]), plt.colorbar(), plt.title("Ground-truth")
plt.figure(), plt.imshow(HSI_bilin[20, :, :]), plt.colorbar(), plt.title("Estimation")

Quantitative results: we now turn towards metric to assess our result. The SNR and PSNR might not be the most adapted metric here because of the large scale factors between the super-resolved image and the ground-truth. In addition, to further take into account that there is a spectrum for each pixel, we will rather turn toward the SAD metric, as what we did in the unmixing part. The function below computes the average SAD over the pixels. Use it to quantitavely evaluate your results (we will comparate with other methods later on).

In [ ]:
def evalSR(HSI0, HSI_est):
    # HSI0: ground-truth HSI (cube)
    # HSI_est: estimated HSI (cube)
    HSI0 = HSI0.reshape(HSI0.shape[0], HSI0.shape[1] * HSI0.shape[2])
    HSI_est = HSI_est.reshape(HSI_est.shape[0], HSI_est.shape[1] * HSI_est.shape[2])

    metriqueTab = np.zeros(HSI0.shape[1])
    for ii in range(HSI0.shape[1]):
        metriqueTab[ii] = np.sum(HSI0[:, ii] * HSI_est[:, ii]) / (
            np.linalg.norm(HSI0[:, ii]) * np.linalg.norm(HSI_est[:, ii])
        )

    return np.mean(np.abs(metriqueTab))

In [ ]:
print(
    "SAD interpolation bilineaire en dB: ",
    -10 * np.log10(np.abs(1 - (evalSR(data, HSI_bilin)))),
)

## 3) Hyperspectral unmixing + bilinear interpolation of the abundances (single image SR)

In this part, we still use bilinear interpolation, but we also leverage first a NMF decomposition to reduce the data dimensionality and obtain $A_{MU\ HSI},S_{MU\ HSI}$. Once the NMF performed, we will use a bilinear interpolation on the abundance maps to obtain $S_{MU\ HSI\ bilin}$, which will be used to synthetise the high resolution HSI: $HSI_{abundance\ bilin} = A_{MU\ HSI}S_{MU\ HSI\ bilin}$.

First step: decompose the HSI using the MU algorithm that you coded in the HSU part. Here, note that you can use a much larger number of endmembers than previously: since we are not directly interested by the endmember interpretability, it is not a big issue if some of them are wrong when we use a large $n$ value.

In [ ]:
HSI.reshape(HSI.shape[0], -1).shape

In [ ]:
n_MU = 30
HSI_int = HSI.reshape(HSI.shape[0], -1)
A_MU_HSI, S_MU_HSI = MULecture(
    HSI_int, n=n_MU, N_Iter=300, tolerance=1e-3, plot_evolution=1
)

In [ ]:
print(A_MU_HSI.shape)
print(S_MU_HSI.shape)

Now that we have $S_{MU\ HSI}$, let us increase its resolution by a facto $w$ using bilinear interpolation. Store the result in S_MU_HSI_bilin.

In [ ]:
S_MU_HSI = S_MU_HSI.reshape([n_MU, HSI.shape[1], HSI.shape[2]])

S_MU_HSI_bilin = np.zeros(
    (S_MU_HSI.shape[0], S_MU_HSI.shape[1] * w, S_MU_HSI.shape[2] * w)
)

for ii in range(S_MU_HSI.shape[0]):
    S_MU_HSI_bilin[ii, :, :] = zoom_bi(S_MU_HSI[ii, :, :], w)
S_MU_HSI_bilin.shape

Look at the difference between a few maps and S_MU_HSI and their interpolated versions. What do you observe?


__answer__:
The comparison between the original \( S_{MU\ HSI} \) abundance maps and their bilinearly interpolated versions shows that the interpolation increases the spatial resolution, making the maps smoother and less sharp. The bilinear interpolation introduces a continuous transition between adjacent values, leading to a loss of high-frequency details and blurring fine variations present in the original data. Consequently, the interpolated abundance maps appear more homogeneous, with features becoming less distinct, and some of the abrupt boundaries or transitions from the original maps are softened or lost.

In [ ]:
(
    plt.figure(),
    plt.imshow(S_MU_HSI[8, :, :]),
    plt.colorbar(),
    plt.title("Original abundance"),
)
(
    plt.figure(),
    plt.imshow(S_MU_HSI_bilin[8, :, :]),
    plt.colorbar(),
    plt.title("Interpolated abundance"),
)

Second step: synthetise the high resolution HSI by matrix multiplication: $HSI_{abundance\ bilin} = A_{MU\ HSI}S_{MU\ HSI\ bilin}$. Qualitatively Look at your results and compare to the ones obtained in the previous subsection.

In [ ]:
S_MU_HSI_bilin_reshaped = S_MU_HSI_bilin.reshape(S_MU_HSI_bilin.shape[0], -1)
HSI_abundance_bilin = np.dot(A_MU_HSI, S_MU_HSI_bilin_reshaped)
HSI_abundance_bilin = HSI_abundance_bilin.reshape(
    A_MU_HSI.shape[0], S_MU_HSI_bilin.shape[1], S_MU_HSI_bilin.shape[2]
)

In [ ]:
plt.figure(), plt.imshow(data[20, :, :]), plt.colorbar(), plt.title("Ground-truth")
(
    plt.figure(),
    plt.imshow(HSI_bilin[60, :, :]),
    plt.colorbar(),
    plt.title("Estimation by direct bilinear interpolation"),
)
(
    plt.figure(),
    plt.imshow(HSI_abundance_bilin[60, :, :]),
    plt.colorbar(),
    plt.title("Estimation by bilinear interpolation on the abundance maps"),
)

In [ ]:
(plt.figure(),)
(plt.imshow(HSI_bilin[60, :, :] - HSI_abundance_bilin[60, :, :]),)
(plt.colorbar(),)
plt.title("Difference between the two methods")

__answer__:Overall, the differences are very subtle. Some specific parts of the image (e.g., the bottom) exhibit minor visual differences, but these are almost invisible to the human eye. This suggests that the new method does not seem to be working effectively.


Lastly, look at the quantitative results. Is this new method working well? Why? Try with different number of endmembers.


__answer__: 
We obtained similar dB values (21.9 for the first method vs. 21.7 for the new method). This aligns with our visual observations, confirming that the new method does not provide a significant improvement.  

This result can be easily explained by the fact that the NMF (MU) decomposition is a linear process and does not significantly alter the quality of the interpolated values. Consequently, the reconstruction remains nearly identical to direct bilinear interpolation.  

When increasing the number of endmembers, both methods produce almost the exact same abundance maps. This indicates that modifying the number of endmembers does not significantly impact the overall performance of the algorithm. The core limitation likely lies in the interpolation step rather than the spectral decomposition itself.






In [ ]:
print(
    "SAD interpolation bilineaire of abundances en dB: ",
    -10 * np.log10(np.abs(1 - evalSR(data, HSI_abundance_bilin))),
)

In [ ]:
n_MU = 70
HSI_int = HSI.reshape(HSI.shape[0], -1)
A_MU_HSI, S_MU_HSI = MULecture(
    HSI_int, n=n_MU, N_Iter=300, tolerance=1e-3, plot_evolution=1
)
S_MU_HSI = S_MU_HSI.reshape([n_MU, HSI.shape[1], HSI.shape[2]])

S_MU_HSI_bilin = np.zeros(
    (S_MU_HSI.shape[0], S_MU_HSI.shape[1] * w, S_MU_HSI.shape[2] * w)
)

for ii in range(S_MU_HSI.shape[0]):
    S_MU_HSI_bilin[ii, :, :] = zoom_bi(S_MU_HSI[ii, :, :], w)
S_MU_HSI_bilin.shape
(
    plt.figure(),
    plt.imshow(S_MU_HSI[8, :, :]),
    plt.colorbar(),
    plt.title("Original abundance"),
)
(
    plt.figure(),
    plt.imshow(S_MU_HSI_bilin[8, :, :]),
    plt.colorbar(),
    plt.title("Interpolated abundance"),
)
S_MU_HSI_bilin_reshaped = S_MU_HSI_bilin.reshape(S_MU_HSI_bilin.shape[0], -1)
HSI_abundance_bilin = np.dot(A_MU_HSI, S_MU_HSI_bilin_reshaped)
HSI_abundance_bilin = HSI_abundance_bilin.reshape(
    A_MU_HSI.shape[0], S_MU_HSI_bilin.shape[1], S_MU_HSI_bilin.shape[2]
)
plt.figure(), plt.imshow(data[20, :, :]), plt.colorbar(), plt.title("Ground-truth")
(
    plt.figure(),
    plt.imshow(HSI_bilin[60, :, :]),
    plt.colorbar(),
    plt.title("Estimation by direct bilinear interpolation"),
)
(
    plt.figure(),
    plt.imshow(HSI_abundance_bilin[60, :, :]),
    plt.colorbar(),
    plt.title("Estimation by bilinear interpolation on the abundance maps"),
)
(plt.figure(),)
(plt.imshow(HSI_bilin[60, :, :] - HSI_abundance_bilin[60, :, :]),)
(plt.colorbar(),)
plt.title("Difference between the two methods")

In [ ]:
n_MU = 70
HSI_int = HSI.reshape(HSI.shape[0], -1)
A_MU_HSI, S_MU_HSI = MULecture(
    HSI_int, n=n_MU, N_Iter=300, tolerance=1e-3, plot_evolution=1
)
S_MU_HSI = S_MU_HSI.reshape([n_MU, HSI.shape[1], HSI.shape[2]])

S_MU_HSI_bilin = np.zeros(
    (S_MU_HSI.shape[0], S_MU_HSI.shape[1] * w, S_MU_HSI.shape[2] * w)
)

for ii in range(S_MU_HSI.shape[0]):
    S_MU_HSI_bilin[ii, :, :] = zoom_bi(S_MU_HSI[ii, :, :], w)
S_MU_HSI_bilin.shape
(
    plt.figure(),
    plt.imshow(S_MU_HSI[8, :, :]),
    plt.colorbar(),
    plt.title("Original abundance"),
)
(
    plt.figure(),
    plt.imshow(S_MU_HSI_bilin[8, :, :]),
    plt.colorbar(),
    plt.title("Interpolated abundance"),
)
S_MU_HSI_bilin_reshaped = S_MU_HSI_bilin.reshape(S_MU_HSI_bilin.shape[0], -1)
HSI_abundance_bilin = np.dot(A_MU_HSI, S_MU_HSI_bilin_reshaped)
HSI_abundance_bilin = HSI_abundance_bilin.reshape(
    A_MU_HSI.shape[0], S_MU_HSI_bilin.shape[1], S_MU_HSI_bilin.shape[2]
)
plt.figure(), plt.imshow(data[20, :, :]), plt.colorbar(), plt.title("Ground-truth")
(
    plt.figure(),
    plt.imshow(HSI_bilin[60, :, :]),
    plt.colorbar(),
    plt.title("Estimation by direct bilinear interpolation"),
)
(
    plt.figure(),
    plt.imshow(HSI_abundance_bilin[60, :, :]),
    plt.colorbar(),
    plt.title("Estimation by bilinear interpolation on the abundance maps"),
)
(plt.figure(),)
(plt.imshow(HSI_bilin[60, :, :] - HSI_abundance_bilin[60, :, :]),)
(plt.colorbar(),)
plt.title("Difference between the two methods")

# 4) Simple HSI super-resolution with non-blind fusion

To do better than the two above methods, we will try to introduce the space information contained in the available MSI and fuse it with the HSI to obtain a SR HSI image. To do that, we will have a simple approach: first perform NMF on the HSI, obtaining $A_{MU\ HSI}$ and $S_{MU\ HSI}$ (this was already done in the last subsection). Then, we will also decompose the MSI into $A_{MU\ MSI}$ and $S_{MU\ MSI}$. Since $A_{MU\ HSI}$ and $S_{MU\ MSI}$ are expected to be of good quality, we will fuse them to obtain the HR-SR image: $HSI_{fusion} = A_{MU\ HSI}S_{MU\ MSI}$.

A difficulty is however that performing unmixing on the MSI is highly difficult since there are few spectral bands in this image. As such, a fully blind approach in which we apply an NMF decomposition to the MSI would not work well (you can have a look at the last optional part of this practival work if you want to test that). Consequently, it is better to help the unmixing of the MSI with the one of the HSI.

To do that, it is possible to estimate the MSI endmembers from the ones found with the HSI (which are of good quality) using the spectral response function (SRF) that we assume to be known. Mathematically: $A_{MU\ MSI} = SRF\times A_{MU\ HSI}$.

Then, $S_{MU\ MSI}$ can be obtained by solving a nonnegative least-square problem:
\begin{equation}
\arg\min_{S_{MU\ MSI}} \frac{1}{2}\|MSI - A_{MU\ MSI}S_{MU\ MSI}\|_F^2,
\end{equation}

(with MSI into matrix form).


In practice, the optimization problem will be solved applying the MU algorithm with a fixed $A_{MU\ HSI}$.

First fill the cell below to compute A_MU_MSI from the SRF and A_MU_HSI.

In [ ]:
A_MU_MSI = np.dot(srf, A_MU_HSI)

print(A_MU_MSI.shape)

Then, compute S_MU_MSI by solving the nonnegative least-square problem with the MU. Note that you can use "frozenW = True" in the function parameters to fix A.

In [ ]:
temp, S_MU_MSI = MULecture(
    MSI.reshape(MSI.shape[0], -1),
    n=A_MU_MSI.shape[1],
    N_Iter=300,
    tolerance=1e-3,
    plot_evolution=1,
    A=A_MU_MSI,
    frozenA=True,
)

# Reshape abundance maps back to spatial dimensions
S_MU_MSI = S_MU_MSI.reshape(S_MU_MSI.shape[0], MSI.shape[1], MSI.shape[2])

Compare the HSI abundances and the MSI abundances. What do you observe?


__answer__:  
The MSI abundances are smoother compared to the HSI abundances. The HSI abundances exhibit clearer spikes and more distinct transitions between materials, whereas the MSI abundances is more blended and continuous. This smoothness in MSI abundances is due to the lower spectral resolution, which results in a loss of fine-grained spectral variations.  

HSI captures more precise material separations, revealing sharper boundaries and detailed local variations. 

In [ ]:
(
    plt.figure(),
    plt.imshow(S_MU_HSI.reshape(n_MU, HSI.shape[1], HSI.shape[2])[3]),
    plt.colorbar(),
    plt.title("HSI abundances"),
)
(
    plt.figure(),
    plt.imshow(S_MU_MSI.reshape(n_MU, MSI.shape[1], MSI.shape[2])[3]),
    plt.colorbar(),
    plt.title("MSI abundances"),
)

Now, generate the fused high-resolution HSI image by multiplying A_MU_HSI and S_MU_MSI.

In [ ]:
S_MU_HSI_reshaped = S_MU_HSI.reshape(S_MU_HSI.shape[0], -1)
HSI_fusion = np.dot(A_MU_HSI, S_MU_HSI_reshaped)
HSI_fusion = HSI_fusion.reshape(A_MU_HSI.shape[0], S_MU_HSI.shape[1], S_MU_HSI.shape[2])

Compare the image obtained through fusion to HSI_abundance_bilin. Comment both the quantitative and qualitative results.

__answer__:

# 4) Simple HSI super-resolution with non-blind fusion

To do better than the two above methods, we will try to introduce the space information contained in the available MSI and fuse it with the HSI to obtain a SR HSI image. To do that, we will have a simple approach: first perform NMF on the HSI, obtaining $A_{MU\ HSI}$ and $S_{MU\ HSI}$ (this was already done in the last subsection). Then, we will also decompose the MSI into $A_{MU\ MSI}$ and $S_{MU\ MSI}$. Since $A_{MU\ HSI}$ and $S_{MU\ MSI}$ are expected to be of good quality, we will fuse them to obtain the HR-SR image: $HSI_{fusion} = A_{MU\ HSI}S_{MU\ MSI}$.

A difficulty is however that performing unmixing on the MSI is highly difficult since there are few spectral bands in this image. As such, a fully blind approach in which we apply an NMF decomposition to the MSI would not work well (you can have a look at the last optional part of this practival work if you want to test that). Consequently, it is better to help the unmixing of the MSI with the one of the HSI.

To do that, it is possible to estimate the MSI endmembers from the ones found with the HSI (which are of good quality) using the spectral response function (SRF) that we assume to be known. Mathematically: $A_{MU\ MSI} = SRF\times A_{MU\ HSI}$.

Then, $S_{MU\ MSI}$ can be obtained by solving a nonnegative least-square problem:
\begin{equation}
\arg\min_{S_{MU\ MSI}} \frac{1}{2}\|MSI - A_{MU\ MSI}S_{MU\ MSI}\|_F^2,
\end{equation}

(with MSI into matrix form).


In practice, the optimization problem will be solved applying the MU algorithm with a fixed $A_{MU\ HSI}$.

First fill the cell below to compute A_MU_MSI from the SRF and A_MU_HSI.

In [ ]:
A_MU_MSI = np.dot(srf, A_MU_HSI)

print(A_MU_MSI.shape)

Then, compute S_MU_MSI by solving the nonnegative least-square problem with the MU. Note that you can use "frozenW = True" in the function parameters to fix A.

In [ ]:
temp, S_MU_MSI = MULecture(
    MSI.reshape(MSI.shape[0], -1),
    n=A_MU_MSI.shape[1],
    N_Iter=300,
    tolerance=1e-3,
    plot_evolution=1,
    A=A_MU_MSI,
    frozenA=True,
)

# Reshape abundance maps back to spatial dimensions
S_MU_MSI = S_MU_MSI.reshape(S_MU_MSI.shape[0], MSI.shape[1], MSI.shape[2])

Compare the HSI abundances and the MSI abundances. What do you observe?


__answer__:  
The MSI abundances are smoother compared to the HSI abundances. The HSI abundances exhibit clearer spikes and more distinct transitions between materials, whereas the MSI abundances is more blended and continuous. This smoothness in MSI abundances is due to the lower spectral resolution, which results in a loss of fine-grained spectral variations.  

HSI captures more precise material separations, revealing sharper boundaries and detailed local variations. 

In [ ]:
(
    plt.figure(),
    plt.imshow(S_MU_HSI.reshape(n_MU, HSI.shape[1], HSI.shape[2])[3]),
    plt.colorbar(),
    plt.title("HSI abundances"),
)
(
    plt.figure(),
    plt.imshow(S_MU_MSI.reshape(n_MU, MSI.shape[1], MSI.shape[2])[3]),
    plt.colorbar(),
    plt.title("MSI abundances"),
)

Now, generate the fused high-resolution HSI image by multiplying A_MU_HSI and S_MU_MSI.

In [ ]:
S_MU_MSI_reshaped = S_MU_MSI.reshape(S_MU_MSI.shape[0], -1)
HSI_fusion = np.dot(A_MU_HSI, S_MU_MSI_reshaped)
HSI_fusion = HSI_fusion.reshape(A_MU_HSI.shape[0], S_MU_MSI.shape[1], S_MU_MSI.shape[2])

Compare the image obtained through fusion to HSI_abundance_bilin. Comment both the quantitative and qualitative results.


__answer__:

The fused image significantly outperforms the bilinearly interpolated HSI in terms of resolution. The fusion method results in a sharper image with more defined abundance maps, showing distinct peaks. This improvement is reflected in the quantitative results, with a notable increase of +5 compared to the bilinear interpolation.

In [ ]:
L_obs = 100
plt.figure(), plt.imshow(data[L_obs, :, :]), plt.colorbar()
plt.figure(), plt.imshow(HSI_abundance_bilin[L_obs, :, :]), plt.colorbar()
plt.title("Abundance maps bilinear interpolation")
plt.figure(), plt.imshow(HSI_fusion[L_obs, :, :]), plt.colorbar()
plt.title("Abundance maps fusion")

In [ ]:
print("SAD simple fusion: ", -10 * np.log10(np.abs(1 - evalSR(data, HSI_fusion))))

## 4) Coupled nonnegative matrix factorization for HSI and MSI fusion

In this last part, we apply the CNMF algorihtm that we saw in class. Although the principle is quite simple, there are a few subtlties in the code. Therefore, everything is already implemented. You just have to launch the code and comment the results. What do you think might explain the differences between your results and the ones of CNMF?

__answer__:
The differences between the results obtained from CNMF and other methods can be attributed to the fact that CNMF incorporates information from both the HSI and MSI simultaneously, allowing for more accurate separation of endmembers and abundance estimation. The joint optimization of both datasets leads to better results, but also introduces complexities in terms of handling the coupling between the datasets, which might explain the  discrepancies compared to simpler methods proposed before that do not fully exploit this relationship.

In [ ]:
from CNMF import CNMF_fusion

print("Start CNMF ...")
I_CNMF = CNMF_fusion(HSI_res, MSI_res)

In [ ]:
plt.figure(), plt.plot(data[:, 120, :])
plt.figure(), plt.plot(I_CNMF.transpose(2, 0, 1)[:, 120, :])

In [ ]:
print(
    "Coupled NMF results",
    -10 * np.log10(np.abs(1 - evalSR(data, I_CNMF.transpose((2, 0, 1))))),
)

## 5) Simple MSI and HSI fusion - optional !

This optional part aims at performing a naive blind fusion between the MSI and HSI, without taking into account the link between both through the SRF. The objective is to show the limitations of such an approach. To do that, perform NMF on the HSI, obtaining $A_{MU\ HSI}$ and $S_{MU\ HSI}$ (this was already done in the subsection 2) and do the same on the MSI, obtaining $A_{MU\ MSI}$ and $S_{MU\ MSI}$. Since $A_{MU\ HSI}$ and $S_{MU\ MSI}$ are expected to be of good quality, we will fuse them to obtain the HR-SR image: $HSI_{fusion} = A_{MU\ HSI}S_{MU\ MSI}$.

In [ ]:
A_MU_MSI, S_MU_MSI = MULecture(
    MSI.reshape(MSI.shape[0], -1), n=n_MU, N_Iter=300, tolerance=1e-3, plot_evolution=1
)
A_MU_HSI, S_MU_HSI = MULecture(
    HSI_int, n=n_MU, N_Iter=300, tolerance=1e-3, plot_evolution=1
)

In [ ]:
(
    plt.figure(),
    plt.imshow(S_MU_HSI.reshape(n_MU, HSI.shape[1], HSI.shape[2])[3]),
    plt.colorbar(),
    plt.title("HSI abundances"),
)
(
    plt.figure(),
    plt.imshow(S_MU_MSI.reshape(n_MU, MSI.shape[1], MSI.shape[2])[3]),
    plt.colorbar(),
    plt.title("MSI abundances"),
)

First perform unmixing on the MSI. Use the same number of endmembers as previously.

Have a look to the S_MU_MSI. Any observation? Can you compare easily the abundance maps?
__answer__: the S_MU_MSI looks similar to the previous S_Mu we computed. It is low obversation and we can't compute it easily with the abundance map.


If you try to directly multiply $A_{MU\ HSI}$ and $S_{MU\ MSI}$, you will obtain bad results. Why is it the case? Hint: remember that there is a scaling and permutation indeterminacy when using MU. Explain how to bypass this issue!

__answer__:
When directly multiplying $ A_{MU\ HSI} $ and $ S_{MU\ MSI} $, incorrect results arise due to scaling and permutation indeterminacies inherent in the Multiplicative Update (MU) algorithm. The scaling indeterminacy occurs because the algorithm does not uniquely define the magnitudes of the elements in $ A $ and $ S $, allowing for arbitrary scaling of the matrices. The permutation indeterminacy arises because the order of columns in $ A $ and rows in $ S $ may differ across datasets, leading to misalignment of endmembers. To bypass these issues, we correct the permutation using an optimization algorithm like the Munkres algorithm, which aligns the endmembers across the datasets, and normalize the abundance maps in $ S_{MU\ MSI} $ to match the scaling of $ S_{MU\ HSI} $, ensuring accurate fusion of the data.







The correcPerm function enables to correct for the permutations. In addition, necessary normalization is performed. You just need to fill the final generation of the HR-SR image. Do it and comment your results, both from a qualitative and quantitative point of view.

In [ ]:
WPerm, indPerm = correctPerm(S_MU_HSI_bilin.reshape(n_MU, 240 * 240).T, S_MU_MSI.T)

In [ ]:
S_MU_MSI_norm_perm = S_MU_MSI[indPerm, :]
for ii in range(S_MU_MSI.shape[0]):
    S_MU_MSI_norm_perm[ii, :] = (
        S_MU_MSI_norm_perm[ii, :]
        / np.linalg.norm(S_MU_MSI_norm_perm[ii, :])
        * np.linalg.norm(S_MU_HSI_bilin[ii, :, :].reshape(240 * 240))
    )

HSI_fusion = np.dot(A_MU_HSI, S_MU_MSI_norm_perm)

HSI_fusion = HSI_fusion.reshape(HSI.shape[0], MSI.shape[1], MSI.shape[2])

In [ ]:
L_obs = 100
plt.figure(), plt.imshow(data[L_obs, :, :]), plt.colorbar()
plt.figure(), plt.imshow(HSI_abundance_bilin[L_obs, :, :]), plt.colorbar()
plt.figure(), plt.imshow(HSI_fusion[L_obs, :, :]), plt.colorbar()

In [ ]:
print("SAD simple fusion: ", -10 * np.log10(np.abs(1 - evalSR(data, HSI_fusion))))

Conclude on the importance to take into account the SRF when doing super-resolution through fusion.


__answer__: When performing super-resolution through fusion, it is crucial to account for the Spectral Response Function (SRF) because it directly influences the relationship between the spectral bands of the hyperspectral (HSI) and multispectral (MSI) images. The SRF defines how the spectral bands of the MSI correspond to the broader spectral bands of the HSI, and without considering this relationship, the fusion process can yield inaccurate results. By incorporating the SRF, we ensure that the endmembers from the HSI and MSI are properly aligned, and the spectral content of the MSI can be properly used to enhance the spatial resolution of the HSI. Ignoring the SRF can lead to poor fusion, where the spectral characteristics of the MSI may not be properly mapped to the HSI, ultimately reducing the quality of the super-resolved image.

